# PS3: Address Geocoder (Demo/Lightweight Pipeline)

This notebook builds a simple geocoder that learns from successful field visits.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import os

print("Packages loaded successfully.")

## 2. Data Loading

In [ ]:
base_path = r"../../Dataset/data"
print("Setting up base paths...")

In [ ]:
addresses = pd.read_csv(os.path.join(base_path, "shared", "addresses.csv"))
baseline = pd.read_csv(os.path.join(base_path, "ps3_geocoder", "baseline_geocodes.csv"))
print(f"Addresses loaded: {len(addresses)}")
print(f"Baseline geocodes loaded: {len(baseline)}")

In [ ]:
field_visits = pd.read_csv(os.path.join(base_path, "shared", "field_visits.csv"))
surveyed = pd.read_csv(os.path.join(base_path, "ps3_geocoder", "surveyed_addresses.csv"))
print(f"Field visits loaded: {len(field_visits)}")
print(f"Surveyed truth data loaded: {len(surveyed)}")

print("\nSample of field visits:")
print(field_visits.head())

## 3. Data Processing

In [ ]:
print("Filtering for successful visits to estimate coordinates...")
success_outcomes = ['cash_collected', 'ptp_collected', 'met_borrower']
successful_visits = field_visits[field_visits['outcome'].isin(success_outcomes)]
print(f"Found {len(successful_visits)} successful visits out of {len(field_visits)} total.")

In [ ]:
print("Calculating mean coordinates per address based on successful visits...")
address_gps = successful_visits.groupby('address_id').agg(
    visit_x=('checkin_x', 'mean'),
    visit_y=('checkin_y', 'mean')
).reset_index()
print(f"Generated coordinate estimates for {len(address_gps)} addresses.")
print(address_gps.head())

In [ ]:
print("Merging with baseline geocoder...")
data = pd.merge(addresses, baseline, on='address_id', how='left')
data = pd.merge(data, address_gps, on='address_id', how='left')

In [ ]:
print("Setting up predictions: Use visit GPS if available, else fallback to baseline geocoder.")
data['pred_x'] = data['visit_x'].fillna(data['geocoder_x'])
data['pred_y'] = data['visit_y'].fillna(data['geocoder_y'])

print(f"Predictions mapped. Missing values remaining: {data['pred_x'].isna().sum()}")

## 4. Evaluation

In [ ]:
print("Evaluating against surveyed truth data...")
eval_data = pd.merge(surveyed, data, on='address_id', how='inner')
print(f"Evaluation dataset shape: {eval_data.shape}")

In [ ]:
print("Calculating euclidean errors...")
eval_data['model_error'] = np.sqrt((eval_data['pred_x'] - eval_data['surveyed_x'])**2 + (eval_data['pred_y'] - eval_data['surveyed_y'])**2)

In [ ]:
print("\n==============================")
print("Lightweight Model Metrics (m):")
print("==============================")
print(f"Median Error:    {eval_data['model_error'].median():.2f}")
print(f"90th Percentile: {eval_data['model_error'].quantile(0.90):.2f}")
print("==============================")